# Task 4: Model Development

This notebook trains and evaluates baseline machine learning models for the credit risk proxy target created during feature engineering.

In [1]:
import sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

project_root = Path.cwd().parent
sys.path.append(str(project_root))

from src.data_processing import load_data
from src.modeling import (
    prepare_model_data,
    split_data,
    train_logistic_regression,
    train_random_forest,
    evaluate_model,
    get_confusion_matrix,
    get_classification_report
)

In [2]:
data_path = "../data/processed/feature_engineered_data.csv"

df = load_data(data_path)

df.head()

,TransactionId,BatchId,AccountId,SubscriptionId,CustomerId,CurrencyCode,CountryCode,ProductId,Amount,Value,...,ProductCategory_ticket,ProductCategory_transport,ProductCategory_tv,ProductCategory_utility_bill,ChannelId_ChannelId_2,ChannelId_ChannelId_3,ChannelId_ChannelId_5,PricingStrategy_1,PricingStrategy_2,PricingStrategy_4
0,TransactionId_76871,BatchId_36123,AccountId_3957,SubscriptionId_887,CustomerId_4406,UGX,256,ProductId_10,1000.0,1000,...,False,False,False,False,False,True,False,False,True,False
1,TransactionId_73770,BatchId_15642,AccountId_4841,SubscriptionId_3829,CustomerId_4406,UGX,256,ProductId_6,-20.0,20,...,False,False,False,False,True,False,False,False,True,False
2,TransactionId_26203,BatchId_53941,AccountId_4229,SubscriptionId_222,CustomerId_4683,UGX,256,ProductId_1,500.0,500,...,False,False,False,False,False,True,False,False,True,False
3,TransactionId_380,BatchId_102363,AccountId_648,SubscriptionId_2185,CustomerId_988,UGX,256,ProductId_21,20000.0,21800,...,False,False,False,True,False,True,False,False,True,False
4,TransactionId_28195,BatchId_38780,AccountId_4841,SubscriptionId_3829,CustomerId_988,UGX,256,ProductId_6,-644.0,644,...,False,False,False,False,True,False,False,False,True,False


In [3]:
df.shape

(95662, 32)

In [4]:
df["is_high_risk"].value_counts()

is_high_risk
0    84653
1    11009
Name: count, dtype: int64

## Modeling Dataset Overview

The feature-engineered dataset created in Task 3 is used for model development. The dataset contains 95,662 records and 32 engineered features after encoding categorical variables.

The target variable is `is_high_risk`, which represents the proxy credit risk label generated from RFM-based customer segmentation.

The target distribution shows a moderate class imbalance, with lower-risk customers forming the majority class. This imbalance is considered during model training and evaluation.

In [5]:
X, y = prepare_model_data(df, target_column="is_high_risk")

X.shape, y.shape

((95662, 22), (95662,))

In [6]:
X.head()

,Amount,Value,FraudResult,ProviderId_ProviderId_2,ProviderId_ProviderId_3,ProviderId_ProviderId_4,ProviderId_ProviderId_5,ProviderId_ProviderId_6,ProductCategory_data_bundles,ProductCategory_financial_services,...,ProductCategory_ticket,ProductCategory_transport,ProductCategory_tv,ProductCategory_utility_bill,ChannelId_ChannelId_2,ChannelId_ChannelId_3,ChannelId_ChannelId_5,PricingStrategy_1,PricingStrategy_2,PricingStrategy_4
0,1000.0,1000,0,False,False,False,False,True,False,False,...,False,False,False,False,False,True,False,False,True,False
1,-20.0,20,0,False,False,True,False,False,False,True,...,False,False,False,False,True,False,False,False,True,False
2,500.0,500,0,False,False,False,False,True,False,False,...,False,False,False,False,False,True,False,False,True,False
3,20000.0,21800,0,False,False,False,False,False,False,False,...,False,False,False,True,False,True,False,False,True,False
4,-644.0,644,0,False,False,True,False,False,False,True,...,False,False,False,False,True,False,False,False,True,False


In [7]:
X_train, X_test, y_train, y_test = split_data(X, y)

X_train.shape, X_test.shape, y_train.shape, y_test.shape

((76529, 22), (19133, 22), (76529,), (19133,))

## Train-Test Split

The dataset was split into training and testing sets using stratified sampling. Stratification helps preserve the target class distribution in both sets, which is important because credit risk datasets may contain class imbalance.

The final split produced:
- 76,529 training samples
- 19,133 testing samples

A total of 22 input features were used for model training.